# 1. Configuración base

In [ ]:
import sys
from pathlib import Path

# Funciona desde la ra?z del taller, la del proyecto o notebooks/.
carpeta = Path.cwd().resolve()
raiz = next((p for base in (carpeta, *carpeta.parents)
             for p in (base, base / "agente-viajes-a2a")
             if (p / "notebook_runtime.py").is_file()), None)
if raiz is None:
    raise RuntimeError("Abre el notebook dentro de la carpeta del proyecto.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))
from notebook_runtime import asegurar_servidor
A2A_URL = asegurar_servidor()
print(f"Servidor disponible: {A2A_URL}")
import os, uuid
A2A_VIAJES_URL = A2A_URL
A2A_VIAJES_GRAPH = os.environ["A2A_VIAJES_GRAPH"]
print(A2A_VIAJES_URL, A2A_VIAJES_GRAPH)

In [ ]:
import httpx
r = httpx.post(f"{A2A_VIAJES_URL}/assistants/search", json={"graph_id": A2A_VIAJES_GRAPH}, timeout=10)
r.raise_for_status()
data = r.json()
assistants = data if isinstance(data, list) else data.get("assistants", [])
if not assistants:
    raise RuntimeError("No hay asistentes para el grafo configurado.")
assistant_id = assistants[0]["assistant_id"]
print(assistant_id)

In [ ]:
import httpx
card = httpx.get(f"{A2A_VIAJES_URL}/.well-known/agent-card.json?assistant_id={assistant_id}", timeout=10).json()
print(card["name"], card["url"])

In [ ]:
import uuid, httpx
context_id = str(uuid.uuid4())
body = {"jsonrpc":"2.0","id":"1","method":"message/send","params":{"message":{"role":"user","messageId":uuid.uuid4().hex,"contextId":context_id,"parts":[{"kind":"text","text":"Quiero un vuelo de Santiago a Lima el 2026-11-10"}]}}}
resp = httpx.post(f"{A2A_VIAJES_URL}/a2a/{assistant_id}", json=body, timeout=90).json()
print(resp["result"]["status"]["state"], context_id)

# 2. Agentes

In [ ]:
from src.consumidor.graph import graph, precio_btc, iniciar_conversacion
from src.consumidor.prompts import SYSTEM_CONVERSACIONAL
print(graph.name); print(bool(SYSTEM_CONVERSACIONAL)); print(precio_btc.invoke({}))

# 3. Consumo

In [ ]:
from langchain_core.messages import HumanMessage
cid = iniciar_conversacion()
config = {"configurable": {"thread_id": "nb-demo-consumidor"}}
def texto(msg):
    c = msg.content
    return c if isinstance(c, str) else " ".join(p.get("text","") for p in c if isinstance(p,dict))
r = await graph.ainvoke({"messages": [HumanMessage(content="Quiero un vuelo de Santiago a Lima el 2026-11-10")]}, config)
print(texto(r["messages"][-1]))

In [ ]:
r = await graph.ainvoke({"messages": [HumanMessage(content="¿Y cuánto está el precio de BTC?")]}, config)
print(texto(r["messages"][-1]))

In [ ]:
r = await graph.ainvoke({"messages": [HumanMessage(content="¿Cuánto salía el vuelo que me buscaste?")]}, config)
print(texto(r["messages"][-1]))